# StreamForge session engineering

You work as a junior data analyst at StreamForge, a fast-growing streaming platform operating across multiple countries.

<img src ="https://data-analytics-fullstack-assets.s3.eu-west-3.amazonaws.com/M05-Exploratory_Data_Analysis/M3_D2_Streamforge.png"/>

The content and growth teams want to validate engagement dashboards and completion metrics without touching real user data. Compliance rules block access to production logs.

They ask you to transform synthetic platform logs into a clean session dataset that reflects how people actually watch content—by title, by genre, by country, and by plan.

You will use Python, NumPy, and Pandas to build one canonical table from multiple sources:

* `user_profiles.csv` with demographics and plan data
* `content.csv` with title metadata
* `viewing_events.csv` with granular play and complete events
* `streaming_events.csv` with raw app-level pings that help you sanity-check durations

<Note type ="important" >

Dowload the CSV files from resources section.

</Note>

## Your mission

Your goal is to produce a session-level DataFrame with full referential integrity, realistic metrics, and defensible cleaning rules. You will compute watched time and completion rate, enrich with user and content attributes, and answer operational questions that product managers use every day.

## Task 1: Load and standardize sources

1. Define a function `load_and_standardize` to load the four CSV files with Pandas and convert time fields to timezone-aware `datetime`.
2. Standardize column names to snake\_case. 
3. Cast numeric fields to the right types so downstream arithmetic behaves predictably.
4. Check keys and confirm that every `user_id` and `content_id` in events exists in their reference tables.


<Note type="hint" title="Loading, parsing, and type safety">

Read CSVs with datetime parsing:
   * Use `pd.read_csv(..., parse_dates=[...])` for fields named like `timestamp`.
   * If some time fields don’t parse on read, convert them with `pd.to_datetime(..., errors="coerce")`.

Lowercase all headers and convert any non-alphanumeric run to underscores: `df.columns = df.columns.str.lower().str.replace(r"[^a-z0-9]+","_", regex=True)`

Timezone normalization: Convert all parsed timestamps to UTC: `df["timestamp"] = pd.to_datetime(df["timestamp"], utc=True)`

Coerce numerics:
   * For duration fields: `pd.to_numeric(..., errors="coerce")` to avoid string math issues.
   * Check expected numeric columns (e.g., `watched_seconds`, `total_seconds`, `duration_seconds`).

Referential integrity checks:
   * Verify that every `user_id` and `content_id` present in events exists in `user_profiles` and `content`.
   * Use set differences to report missing keys. Print counts and a small sample for debugging.

</Note>


In [50]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings("ignore") # Suppress warnings for cleaner output

In [51]:
# Convert timestamp columns and standardize timezone
def load_and_standardize(path, date_cols=None):
    # par défaut date_cols=None, important pour les fichiers sans colonnes de dates
    """
    Load a CSV file into a pandas DataFrame and standardize column names.
    
    Parameters:
    - path (str): Path to the CSV file.
    - date_cols (list of str, optional): List of columns to parse as dates.
    
    Returns:
    - pd.DataFrame: Loaded and standardized DataFrame.
    """
    df = pd.read_csv(path, parse_dates=date_cols)
    df.columns = (
        df.columns.str.lower()
        .str.replace(r"[^a-z0-9]+", "_", regex=True)
        # je remplace toute séquence de caractères non alphanumériques par un _
        # + pour indiquer une séquence de un ou plusieurs caractères (plusieurs occurences)
        .str.strip("_")
        # pour enlever les _ en début ou fin de nom de colonne
    )
    
    
     # Convert datetime columns to UTC
    if date_cols: # only if date_cols=[timestamp] is provided
        for col in date_cols:
            df[col] = pd.to_datetime(df[col], utc=True, errors="coerce")
    
      # Convert numeric columns robustly
    for col in df.select_dtypes(include=["object"]).columns:
        # comme pandas convertit les nombres en object s'ils
        # contiennent des valeurs manquantes ou des chaînes non numériques 
        # je précise include = ["object"]
        if any(df[col].astype(str).str.match(r"^\d+(\.\d+)?$")): # 12 => "12.64" =<  12.64
            # any si au moins une valeur correspond au format numérique
            # astype(str) je convertis d'abord tout en string pour éviter les erreurs
            # sur les NaN, None ou N/A ;
            # puis match avec r"^\d+(\.\d+)?$" qui signifie :
            # début de chaîne (^), un ou plusieurs chiffres (\d+),
            # optionnellement suivis d'un point et d'un ou plusieurs chiffres ((\.\d+)?),
            # fin de chaîne ($)
            df[col] = pd.to_numeric(df[col], errors="ignore")
            # errors="ignore" pour éviter les erreurs et laisser les 
            # autres valeurs telles quelles
            # si la conversion échoue
            # to_numeric convertit en int ou float selon les données
            
    return df  # attention à la bonne indentation!

In [52]:
user_profiles = load_and_standardize("user_profiles.csv")
content = load_and_standardize("content.csv")
viewing_events = load_and_standardize("viewing_events.csv", date_cols=["timestamp"])
streaming_events = load_and_standardize("streaming_events.csv", date_cols=["timestamp"])


In [53]:
# Check referential integrity
# table principale viewing_events
# tables référencées user_profiles et content
missing_users = set(viewing_events.user_id) - set(user_profiles.user_id)

missing_content = set(viewing_events.content_id) - set(content.content_id)

print("Missing users from viewing_events in user_profiles:", len(missing_users))
print("Missing content from missing_contents in user_profiles:", len(missing_content))


Missing users from viewing_events in user_profiles: 0
Missing content from missing_contents in user_profiles: 0


## Task 2: Build a clean “watch sessions” table

1. Define a session as a user’s watch attempt for a single `content_id` on a single calendar day.
2. Aggregate `viewing_events.csv` to one row per `user_id`–`content_id`–`date` by summing `watched_seconds` and taking the max of `total_seconds`.
3. Compute `completion_rate` as `watched_seconds / total_seconds` and clip the result to `[0, 1]`.
4. Merge the session table with `content.csv` to bring `genre`, `primary_genre`, `content_type`, and `rating`.
5. Merge with `user_profiles.csv` to bring `subscription_type`, `user_age`, `country`, and `monthly_revenue`.
6. Create `duration_bucket` by cutting `total_seconds` into quantiles so you can compare movies and series without the length bias. Use `q=5` to create 5 buckets with labels=['Very Short', 'Short', 'Medium', 'Long', 'Very Long'].


<Note type="hint" title="Session Definition & Aggregation">

Define a session date: `session_date = viewing_events["timestamp"].dt.date` to group by calendar day.

Aggregate to session level:
   * Group by `["user_id", "content_id", "session_date"]`.
   * `watched_seconds`: sum across events that day (total time actually watched).
   * `total_seconds`: max for that title/day (the canonical length; max is robust if multiple events disagree).

Compute completion:
   * `completion_rate = watched_seconds / total_seconds`.
   * Clip to `[0, 1]` to avoid >100% due to noisy logs.

Enrich with metadata:
   * Join with `content` to get `genre`, `primary_genre`, `content_type`, `rating`.
   * Join with `user_profiles` to get `subscription_type`, `user_age`, `country`, `monthly_revenue`.
   * Normalize plan labels once (e.g., `.str.lower().str.strip()`).

Duration buckets:
   * Use `pd.qcut(total_seconds, q=5, duplicates="drop")` to create length buckets for fair comparisons.
   * Map categories to labels `['Very Short', 'Short', 'Medium', 'Long', 'Very Long']`.

</Note>

In [54]:
# 1. 
viewing_events["session_date"] = viewing_events["timestamp"].dt.date
# je recrée une colonne session_date pour extraire le jour dans le timestamp
# dt pour garantir que l'opération est vectorisée sur toute la colonne timestamp
# et date pour extraire la date sans l'heure
# Pourquoi? éviter que 2 évènements de visionnage le même jour mais à des heures différentes
# soient considérés comme des sessions différentes


In [55]:
# 2. Create session date for grouping
sessions_df = (viewing_events
    .groupby(["user_id", "content_id", "session_date"])
    .agg(
        watched_seconds=("watched_seconds", "sum"),
        total_seconds=("total_seconds", "max")
    ).reset_index()
)

# reset_index() pour remettre user_id, content_id et 
# session_date en colonnes et non en multi_index

In [56]:
# Aggregate to session level: one row per user-content-date


In [57]:
# 3. Calculate completion rate and clip to [0,1]

sessions_df["completion_rate"] = (
    sessions_df["watched_seconds"] / sessions_df["total_seconds"]
).clip(0,1)
# clip(0,1) pour éviter les valeurs négatives ou supérieures à 1

In [58]:
# Merge with content metadata
sessions_df = (
    sessions_df.merge(
        content, on="content_id", how="left"
    )
)

# Merge with user profiles
sessions_df = sessions_df.merge(user_profiles, on="user_id", how="left")


In [59]:
# Create duration buckets using quantiles qcut()
labels = ["Very Short", "Short", "Medium", "Long", "Very Long"]

sessions_df["duration_bucket"] = pd.qcut(
    sessions_df["total_seconds"],
    q=5, # découper l'échantillon en 5 quantiles égaux avec qcut (# cut pour des intervalles fixes # à revoir)
    labels=labels,
   
    duplicates="drop" # si pas assez de donnees pour 5 quantiles distincts
    # on reduit dynamiquement le nombre de quantiles pour éviter des erreurs de plantage
)
print(sessions_df.head())


    user_id   content_id session_date  watched_seconds  total_seconds  \
0  user_001  content_006   2024-02-01             2552           3895   
1  user_001  content_016   2024-02-19              225           2385   
2  user_001  content_020   2024-02-08              619           1871   
3  user_001  content_022   2024-02-28              902           3872   
4  user_001  content_028   2024-02-28              785           4056   

   completion_rate        genre primary_genre content_type  rating  \
0         0.655199  documentary   documentary       series     7.6   
1         0.094340     thriller      thriller  documentary     7.6   
2         0.330839       sci-fi        sci-fi        movie     6.4   
3         0.232955       comedy        comedy       series     7.1   
4         0.193540      romance       romance        movie     9.0   

  subscription_type  user_age country  monthly_revenue duration_bucket  
0           premium        32      DE            13.99            L

## Task 3: Answer four operational questions from product

1. Compute the average completion rate by `primary_genre` and sort from highest to lowest. 

<Note type="hint">

* Group by `primary_genre` and compute the mean of `completion_rate`.
* Also report `session_count` (number of sessions contributing) so small genres don’t mislead.

</Note>

2. Rank titles by total watch time within each country and return the top ten per country. 

<Note type="hint">
* Group by `["country", "content_id"]` and sum `watched_seconds`.
* Rank within each country using `rank(method="first", ascending=False)`.
* Keep rows with rank ≤ 10 and sort by `["country", "rank"]`.
</Note>

3. Identify countries where premium plans underperform on completion relative to basic plans and quantify the gap in percentage points.

<Note type="hint" >

- Filters `sessions_df` to only `subscription_type ∈ {'basic','premium'}`.
- Computes the mean `completion_rate` per `('country','subscription_type')`.
- Unstacks to a country-level table with two columns: `basic` and `premium`.
- Selects countries where `premium < basic`, i.e., Premium underperforms.

</Note>

In [60]:
# 1. Average completion rate by primary genre
genre_stats = (
    sessions_df
    .groupby("primary_genre")
    .agg(
        completion_rate=("completion_rate", "mean"),
        session_count=("completion_rate", "count")
        # count en plus pour éviter des moyennes trompeuses de catégories ayant peu de sessions
    )
    .sort_values("completion_rate", ascending=False) 
).reset_index() # sans reset_index() on garde primary_genre en index, à éviter
print(genre_stats)


  primary_genre  completion_rate  session_count
0         drama         0.258138            103
1       romance         0.240064            128
2        sci-fi         0.239996            162
3        horror         0.221052             94
4        action         0.219030             64
5      thriller         0.205333            218
6        comedy         0.202865            133
7   documentary         0.200805             98


In [61]:
# 2. Top 10 content by total watch time per country
rank_df = (
    sessions_df
    .groupby(["country", "content_id"])
    .agg(watched_seconds=("watched_seconds", "sum"))
    .reset_index()
    # première étape : somme du temps de visionnage sur 2 clés de regroupement : watched_seconds par pays et par contenu
)

rank_df["rank"] = (
    rank_df.groupby("country")["watched_seconds"]
    # rappel syntaxe : df.groupby("col1")["col2"] pour grouper par col1 et appliquer une fonction sur col2
    # différente de agg qui agrège en une seule ligne par groupe
    # watched_seconds ici est la somme totale par pays et par contenu issu de rank_df
    .rank(method="first", ascending=False)
    # 2e étape : classement par pays en fonction du temps de visionnage total
    # method "first" pour attribuer des rangs en fonction de l'ordre d'apparition en cas d'égalité
)

top10 = rank_df[rank_df["rank"] <= 10].sort_values(["country", "rank"])
# filtrer pour ne garder que les top 10 par pays 
# le tri multi-colonnes suit l'ordre des colonnes dans la liste
# tri par pays puis à l'intérieur de chaque pays par rang
print(top10)



    country   content_id  watched_seconds  rank
29       BR  content_035             7333   1.0
23       BR  content_029             6965   2.0
42       BR  content_049             6653   3.0
82       BR  content_092             6512   4.0
81       BR  content_091             6265   5.0
..      ...          ...              ...   ...
450      US  content_006             4522   6.0
461      US  content_020             4139   7.0
489      US  content_061             4075   8.0
498      US  content_074             3796   9.0
464      US  content_023             3337  10.0

[70 rows x 4 columns]


In [ ]:
# 3. Countries where premium plans underperform vs basic plans
subset = sessions_df[sessions_df.subscription_type.isin(["basic", "premium"])]
# filtrer sur la colonne subscription_type pour ne garder que les abonnements basic et premium avec isin
# qui renvoie les valeurs présentes dans la liste fournie
# sessions_df.subscription_type.isin(["basic", "premium"]) renvoie un masque booléen true/false
# ensuite sessions_df[masque] pour filtrer les lignes où le masque est == True
# pas besoin de .loc car filtre uniquement sur les lignes
# équivalent à : subset = sessions_df[(sessions_df.subscription_type == "basic") | (sessions_df.subscription_type == "premium")]
# ou subset = sessions_df.query("subscription_type in ['basic', 'premium']")
# ou subset=[value in ["basic","premium"] for value in sessions_df.subscription_type]

perf = (
    subset
    .groupby(["country", "subscription_type"])
    .completion_rate.mean() # obligé de prendre la moyenne pour les comparaisons entre groupes
    # pareil que groupby(["country", "subscription_type"])["completion_rate"].mean()
    # ou groupby(["country", "subscription_type"]).agg(completion_rate=("completion_rate", "mean"))
    .unstack()
    # rappel : unstack pour transformer les valeurs uniques d'une colonne en colonnes
    # ici pour avoir une colonne par type d'abonnement basic et premium
    # au lieu d'avoir une ligne par pays et par type d'abonnement
)

underperform = perf[perf["premium"] < perf["basic"]]

# tableau des pays où le taux de complétion des abonnés premium est inférieur à celui des abonnés basic
underperform["gap_pp"] = (underperform["basic"] - underperform["premium"]) * 100
print(underperform.sort_values("gap_pp", ascending=False))


subscription_type     basic   premium    gap_pp
country                                        
US                 0.239188  0.216841  2.234705
BR                 0.234241  0.228048  0.619291
JP                 0.230103  0.224401  0.570238


## Task 4: Export and inspect

1. Export `sessions_df` as a CSV with no index.
2. Run `.info()` and `.describe()` to inspect the final table.

<Note type="hint">
Use sample to inspect rows: `.sample(10)` to inspect 10 random rows from the DataFrame.
</Note>

In [45]:
# Export sessions table
sessions_df.to_csv("sessions.csv", index=False)
print("✓ Exported sessions.csv")


✓ Exported sessions.csv


In [46]:
print("Shape:", sessions_df.shape)
print(sessions_df.completion_rate.describe())
sessions_df.sample(10)


Shape: (1000, 15)
count    1000.000000
mean        0.222415
std         0.182617
min         0.006198
25%         0.082054
50%         0.172825
75%         0.314976
max         0.885021
Name: completion_rate, dtype: float64


,user_id,content_id,session_date,watched_seconds,total_seconds,completion_rate,genre,primary_genre,content_type,rating,subscription_type,user_age,country,monthly_revenue,duration_bucket
700,user_037,content_017,2024-01-14,371,2856,0.129902,comedy,comedy,documentary,6.7,premium,46,JP,13.99,Medium
67,user_004,content_081,2024-02-16,1254,6530,0.192037,romance,romance,documentary,9.3,premium,56,JP,8.99,Very Long
485,user_026,content_061,2024-03-01,354,2705,0.130869,sci-fi,sci-fi,movie,6.9,basic,39,CA,13.99,Medium
773,user_040,content_092,2024-03-28,278,4346,0.063967,sci-fi,sci-fi,documentary,7.3,standard,42,JP,17.99,Long
699,user_037,content_010,2024-02-10,575,3766,0.152682,thriller,thriller,movie,7.3,premium,46,JP,13.99,Long
157,user_009,content_012,2024-03-31,101,828,0.121981,sci-fi,sci-fi,series,8.0,basic,58,US,8.99,Very Short
127,user_007,content_034,2024-01-20,401,1797,0.223150,drama,drama,documentary,7.1,premium,68,BR,8.99,Short
168,user_009,content_074,2024-01-13,652,3980,0.163819,thriller,thriller,series,8.1,basic,58,US,8.99,Long
907,user_047,content_010,2024-03-02,1355,5107,0.265322,thriller,thriller,movie,7.3,basic,64,US,13.99,Very Long
937,user_048,content_043,2024-03-26,192,1220,0.157377,comedy,comedy,documentary,9.3,standard,58,FR,13.99,Very Short
